# Extending `vivarium_csu_alzheimers` with risk factors and fertility

*Drafted 2026-05-27.*

Two proposed extensions:

1. **Risk factors** — wire in the GBD CRA risks that have dementia as an affected cause, so we can run prevention-strategy scenarios (e.g. reduce population BMI / smoking exposure and watch incident AD fall).
2. **Age-specific fertility with parent-child linkage** — track how many adult children are available to care for a parent who develops dementia, particularly for Singapore.

This notebook (a) inventories what GBD and vivarium tooling already provides, (b) flags the architectural decisions, and (c) proposes a phased plan. Code cells query the live install so the inventory stays honest.

Use the `vivarium_artifacts` Jupyter kernel (the default `python3` kernel points at a different env).

## Part 1 — GBD risk factors for dementia

### 1.1 What does GBD attach to dementia?

Iterate `gbd_mapping.risk_factors` and pick out the ones whose `affected_causes` include `alzheimers_disease_and_other_dementias`.

In [1]:
from gbd_mapping import risk_factors, causes
import pandas as pd

dementia = "alzheimers_disease_and_other_dementias"

rows = []
for rf in risk_factors:
    if not hasattr(rf, "affected_causes"):
        continue
    if not rf.affected_causes:
        continue
    if any(c.name == dementia for c in rf.affected_causes):
        tmred = rf.tmred
        rows.append(
            {
                "risk": rf.name,
                "rei_id": int(rf.gbd_id),
                "most_detailed": bool(rf.most_detailed),
                "distribution": str(rf.distribution),
                "tmred_min": None if tmred is None else float(tmred.min),
                "tmred_max": None if tmred is None else float(tmred.max),
                "inverted": None if tmred is None else bool(tmred.inverted),
            }
        )

rf_table = pd.DataFrame(rows).sort_values(["most_detailed", "risk"], ascending=[False, True])
rf_table

,risk,rei_id,most_detailed,distribution,tmred_min,tmred_max,inverted
1,ambient_particulate_matter_pollution,86,True,custom,NaN,NaN,None
5,high_alcohol_use,102,True,custom,NaN,NaN,None
12,high_body_mass_index_in_adults,370,True,ensemble,20.00000,22.500000,False
7,high_fasting_plasma_glucose,105,True,ensemble,4.88488,5.301205,False
2,household_air_pollution_from_solid_fuels,87,True,dichotomous,NaN,NaN,None
4,smoking,99,True,custom,NaN,NaN,None
0,air_pollution,85,False,None,NaN,NaN,None
9,all_risk_factors,169,False,None,NaN,NaN,None
11,behavioral_risks,203,False,None,NaN,NaN,None
10,environmental_occupational_risks,202,False,None,NaN,NaN,None


**Reading the table.** `most_detailed=True` rows are the leaf-level exposures that have their own RR/PAF estimates in the CRA. The `False` rows are aggregate risks (`tobacco`, `air_pollution`, `high_body_mass_index`, `metabolic_risks`, `behavioral_risks`, `environmental_occupational_risks`, `all_risk_factors`) — they roll up the most-detailed children and are not directly usable as exposure variables.

**The six exposures we can actually pull for dementia are**:

| risk | distribution | what it looks like in vivarium_inputs |
|---|---|---|
| `high_body_mass_index_in_adults` | ensemble (continuous) | mean BMI + SD by age/sex/year |
| `high_fasting_plasma_glucose` | ensemble (continuous) | mean FPG + SD |
| `high_alcohol_use` | custom (continuous-ish) | g/day, sex-specific |
| `smoking` | custom | current/former/never categories + pack-years |
| `ambient_particulate_matter_pollution` | custom | µg/m³ PM2.5 |
| `household_air_pollution_from_solid_fuels` | dichotomous | proportion exposed |

**Modifiable-risk landscape vs. the Lancet Commission.** The 2024 Lancet Commission on dementia prevention lists 14 modifiable risk factors (the three above plus less education, hearing loss, hypertension in midlife, depression, physical inactivity, social isolation, traumatic brain injury, untreated vision loss, etc.). The GBD CRA only formally attributes burden for the six above — the others are scientifically real but have no GBD-blessed PAF you can pull off the shelf. If we want to model them, we have to bring in RRs from the meta-analyses cited by the Commission and assemble exposures ourselves.

**Recommendation for the first pass:** start with the GBD-six because the data pipeline already exists and the numbers are reproducible. We can layer Lancet-Commission risks on top as a second phase using custom RR/exposure data.

### 1.2 What's already in our artifact, and what's missing?

The current Singapore artifact has *no* risk-factor data. Check:

In [2]:
from vivarium.framework.artifact import Artifact

art = Artifact("/mnt/share/homes/abie/projects/2026/vivarium_csu_alzheimers/singapore.hdf")
risk_keys = [k for k in art.keys if k.startswith("risk_factor.") or "exposure" in k or "relative_risk" in k or "population_attributable_fraction" in k]
print("Risk-related artifact keys (Singapore):", risk_keys or "NONE")
print("Total keys:", len(art.keys))

Risk-related artifact keys (Singapore): ['treatment.relative_risk']
Total keys: 43


Nothing risk-related is loaded. To add a risk to the artifact we'd extend `data/loader.py` with handlers for these keys per risk `<r>`:

- `risk_factor.<r>.exposure`
- `risk_factor.<r>.exposure_standard_deviation` (continuous distributions)
- `risk_factor.<r>.exposure_distribution_weights` (ensemble distributions)
- `risk_factor.<r>.relative_risk`
- `risk_factor.<r>.population_attributable_fraction`
- `risk_factor.<r>.distribution`
- `risk_factor.<r>.categories` (categorical risks)
- `risk_factor.<r>.tmred` and `relative_risk_scalar` (for continuous)

`vivarium_inputs.interface.get_measure(risk_factors.<r>, "<measure>", location)` is the standard fetch. Per the env-drift memo, `get_draws` chains in the artifact env are fragile — but `vivarium_inputs.get_measure` for these standard risk measures is what the rest of the vivarium ecosystem uses, so if anything currently works, this should.

### 1.3 How VPH expects the model to consume risk data

VPH (installed version 4.13.19) provides two reusable components:

**`vivarium_public_health.risks.Risk`** (`base_risk.py:33`) — exposure model.
- Constructor: `Risk("risk_factor.<name>")`
- Reads `risk_factor.<name>.exposure[/_standard_deviation/_distribution_weights]` from the artifact.
- Adds a per-simulant exposure column based on the distribution type (`dichotomous`, `ordered_polytomous`, `unordered_polytomous`, `normal`, `lognormal`, `ensemble`).

**`vivarium_public_health.risks.RiskEffect`** (`effect.py:32`) — links exposure to a target rate.
- Constructor: `RiskEffect("risk_factor.<name>", "cause.<x>.<rate>")`
- Reads `risk_factor.<name>.relative_risk` and `risk_factor.<name>.population_attributable_fraction`.
- Registers value modifiers on the target pipeline so the rate becomes `baseline * (1-PAF) * RR(exposure)`.

The natural target in our model is the **BBBM → MCI** transition rate, since that's the entry into clinically detectable disease (`components/alzheimers.py:29` defines `BBBMTransitionRate`, registered as a `register_rate_producer` at line 46 — `RiskEffect` modifiers should attach to that pipeline by name without code changes to `alzheimers.py`).

Caveat: `BBBMTransitionRate.compute_transition_rate` currently returns `hazard(time_in_state)` from `data/mci_hazard.py`. We need to verify the rate pipeline `joint_paf` and rate-modifier hooks are actually applied to the value coming out of that source — VPH's `RateTransition.compute_transition_rate` does pull `joint_paf` and modifies the rate. Our override at line 54 does not. **First implementation task: refactor `compute_transition_rate` so it goes through the standard PAF/modifier pipeline, or `RiskEffect` will silently no-op.**

### 1.4 Proposed phased plan — risk factors

**Phase R1 — Plumbing (1–2 days).** Pick one risk (suggest **`high_body_mass_index_in_adults`** — continuous ensemble, well-behaved, large effect). Add loader entries in `data/loader.py` and matching keys in `constants/data_keys.py`. Build a Singapore artifact slice and confirm `Risk(...).setup(...)` runs.

**Phase R2 — Make the target rate paf-aware.** Audit `BBBMTransitionRate.compute_transition_rate` (`components/alzheimers.py:54`). Either route the hazard through the standard rate-modifier chain or register an explicit PAF-aware producer. Add a regression test: with `RiskEffect` attached and exposure=TMREL, the BBBM→MCI rate is unchanged vs. baseline (per BLISS — write the red test first).

**Phase R3 — Add the rest of the GBD-six.** Generalize loader to iterate over a `DEMENTIA_RISKS` tuple. Add `Risk` + `RiskEffect` instances to `model_spec.yaml`. The dichotomous/categorical ones (household air pollution, smoking) are simpler; defer FPG/alcohol if scope tightens.

**Phase R4 — Prevention scenarios.** Add a scenario config dimension `prevention_strategy` with values like `none`, `bmi_to_tmrel`, `smoking_cessation_50pct`, etc. Implement as a coverage knob that shifts a fraction of exposed simulants to lower categories / lower continuous values at simulation start (or progressively over a ramp period).

**Phase R5 — Layer Lancet-Commission risks.** Risks not in GBD's CRA (education, hearing, social isolation, depression, TBI) need custom RR + exposure data. Each can be added as another `Risk`/`RiskEffect` pair backed by literature RRs and either GBD covariates (e.g. educational attainment from covariate `mean_years_of_education`) or sentinel external data. Treat each as a small independent project, gated on whether the policy question actually needs it.

**Decision point before Phase R2:** Joint PAF arithmetic. With multiple risks affecting one rate, GBD computes PAFs assuming a mediation/conditional-independence story that doesn't strictly hold (BMI and FPG, BMI and smoking are correlated). VPH's default joint-PAF combiner multiplies `(1 - PAF_i)` across risks. We should decide whether to accept that (standard practice) or implement something fancier (mediated PAFs). Vote: accept the standard for now and note as a methodological caveat.

## Part 2 — Fertility and parent-child linkage

### 2.1 What VPH provides

`vivarium_public_health.population.add_new_birth_cohorts.FertilityAgeSpecificRates` is the right tool:

- Adds state-table columns `last_birth_time` and **`parent_id`** (the child row's `parent_id` is set to the mother simulant's index).
- Each timestep, eligible women (alive, Female, ≥9 months since last birth) are evaluated against the age-specific fertility rate; sampled births create new simulants via `simulant_creator(parent_ids=…)`.
- Data key required: `covariate.age_specific_fertility_rate.estimate`.

**Good news:** the parent linkage we want already exists out of the box. The child row knows its mother's row id. Walking from a mother to her surviving adult children at time *t* is a `pop[pop.parent_id == m].query('alive==alive and age >= 18')` filter.

**Caveat:** mother→child linkage is direct. The reverse (caregiver-count per dementia case) is what the Singapore question needs. That's a derivation, not new column, but it does mean we have to keep both generations in the active population view (don't untrack adult children too early).

### 2.2 The architectural issue: this model isn't a population model

**This is the most important thing in this notebook.** The current `AlzheimersPopulation` (`components/population.py:17`) does **not** initialize a general population. It only creates simulants who are *about to develop BBBM*, via `AlzheimersIncidence.on_time_step` — every step it Poisson-samples how many new BBBM cases there should be and creates exactly that many simulants. There is no underlying healthy population at risk in the model. The 20,000 sims in `model_spec.yaml` are all destined for the disease pathway.

This is a clever cohort-incidence design that gives us a very efficient simulation of the disease process, but it means **the model has no children, no adult non-cases, and no spouses**. Adding `FertilityAgeSpecificRates` on top of the current population component is not coherent: there are no Female simulants representing the at-risk childbearing population for the fertility component to act on.

There are three ways to address this. Pick one — they imply very different amounts of work.

#### Option A — Bolt on a parallel demographic population (recommended)

Add a second population component that initializes a representative cohort (e.g. females currently aged 15–49 + their existing children at sim start) using `population.structure`, runs `FertilityAgeSpecificRates` against it, and ages everyone forward. These simulants are *not* candidates for AD — they exist only to populate the family tree. The existing `AlzheimersIncidence` simulants are the disease cohort.

When a disease-cohort simulant is created, we **assign them a mother and a set of adult children by sampling from the demographic cohort conditional on their age**. This requires writing a custom matching step but avoids overhauling the disease-cohort design.

Pros: keeps the efficient incident-cohort disease engine; family-supply question becomes a sampling problem. Cons: family ties are statistical artifacts rather than emerging from simulated histories; correlations between, e.g., dementia and number-of-surviving-children are imposed rather than discovered.

#### Option B — Full population sim, no incident-cohort trick

Replace `AlzheimersIncidence` + `AlzheimersPopulation` with VPH's `BasePopulation` + `FertilityAgeSpecificRates` + `Mortality`. Disease incidence comes from a standard `RateTransition` on the susceptible state instead of seeded cohort entry. Sample size has to grow ~50–100× for adequate AD case counts (most simulants will never get AD).

Pros: clean, emergent family histories; family-availability statistics are observed not imposed; risk-factor exposures correlate naturally with cases. Cons: large compute cost; throws out months of work tuning the consistent-rates pipeline.

#### Option C — Skip simulated fertility; do an external micro-sim

Compute the joint distribution of (age of dementia onset, number of living adult children, ages of those children) for Singapore *outside* vivarium using GBD fertility + mortality and a simple matrix-population model, and feed that distribution back as a sim covariate.

Pros: very fast; statistical pipeline already understood. Cons: gives us a marginal distribution, not per-simulant draws; can't answer questions like "if dementia treatment extends life by 3 years, do the same children still provide care?"

**Recommendation:** Option A. It buys us the per-simulant story (which is what the Singapore question really wants — "how many of your kids are alive when you get diagnosed?") without rewriting the disease engine.

### 2.3 Is the fertility data available for Singapore?

In [3]:
# Singapore artifact already has live_births_by_sex (a covariate). For age-specific fertility
# we'd need covariate.age_specific_fertility_rate.estimate added to the artifact.
# Quick check that vivarium_inputs can pull it for Singapore.
from gbd_mapping import covariates
asfr = covariates.age_specific_fertility_rate
print("covariate id:", asfr.gbd_id, " name:", asfr.name)
print("by_age:", asfr.by_age, " by_sex:", asfr.by_sex)
# Don't pull data here — just confirm the mapping is real.

covariate id: cov_id(13)  name: age_specific_fertility_rate
by_age: True  by_sex: True


Singapore is a standard GBD location; ASFR is available. The loader change is one entry in `data/loader.py` and one key in `constants/data_keys.py`.

Mortality data for the demographic-cohort component is already in the artifact (`cause.all_causes.cause_specific_mortality_rate`).

### 2.4 Proposed phased plan — fertility

**Phase F1 — Decide between Options A/B/C.** This is a meeting, not a code change. Without that decision the rest is wasted work.

**Phase F2 (assuming A) — Add a `DemographicPopulation` component.** A second population component initialized from `population.structure` for Singapore, with `Mortality` and `FertilityAgeSpecificRates` operating on it. Stratify the existing `AlzheimersIncidence` simulants from these by adding a `population_role` column (`demographic` vs `disease_cohort`).

**Phase F3 — Assign families to disease-cohort simulants.** When a disease-cohort simulant is created at age *a* in year *y*, sample (i) a hypothetical mother's age at simulant's birth from Singapore's historical ASFR for year *y-a*, (ii) the simulant's own completed fertility from the demographic-cohort distribution. Record children as either real demographic-cohort simulants (if available) or synthetic IDs with attached survival curves.

**Phase F4 — Caregiver observer.** New observer that, on each disease-state transition, records the number of living adult children and their ages. Stratify outputs by that number for the question "do simulants with more children survive longer / get tested earlier / receive treatment more often?".

**Phase F5 — Singapore-specific tuning.** Singapore's TFR has collapsed (≈1.0 in 2024); the historical cohort is very different from the modern one. Make sure ASFR is pulled for the *correct historical year* per simulant, not just sim start year.

## Part 3 — Summary recommendation

1. **Risk factors first.** It's a smaller, more contained extension that delivers obvious modeling value (prevention scenarios). It tests our data pipeline against `vivarium_inputs.get_measure` for the new risk keys, which will probably expose more env-drift issues — better to find those early than during the fertility work.

2. **Make `BBBMTransitionRate` PAF-aware** as a precondition. Without that fix, `RiskEffect` looks like it's working but isn't.

3. **Start with `high_body_mass_index_in_adults`** as the pilot risk. Largest dementia PAF among the GBD-six, well-behaved continuous distribution, exists in `vivarium_inputs` already.

4. **Decide Options A/B/C for fertility before writing any fertility code.** This is a 30-minute conversation that determines whether the work is 1 week or 1 quarter.

5. **Recommend Option A**: parallel demographic cohort + family-tree sampling for disease-cohort simulants. Preserves the consistent-rates work and gives per-simulant family supply for the Singapore caregiving question.

### Concrete next-week deliverables (if approved)

- [ ] Failing test: `RiskEffect` attached to the BBBM→MCI transition with RR=2.0 doubles the rate. (Will fail because `BBBMTransitionRate` bypasses the rate modifier pipeline.)
- [ ] Fix `BBBMTransitionRate.compute_transition_rate` to honor rate modifiers.
- [ ] Add `loader.load_risk_factor_data` and the 5 risk-related data keys for `high_body_mass_index_in_adults`.
- [ ] Rebuild `singapore.hdf` with BMI exposure + RR + PAF; spot-check shape against published Singapore values.
- [ ] Model spec: add `Risk('risk_factor.high_body_mass_index_in_adults')` + `RiskEffect(...)` on the BBBM→MCI rate.
- [ ] One-page memo on Option A/B/C with sample-size estimates.